In [ ]:
import json, os, sys
from typing import Optional
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
client = OpenAI()
import functools
import time
import asyncio
from queue import Queue
from concurrent.futures import as_completed, ThreadPoolExecutor


In [ ]:
system_prompt = """You are an expert in automated planning. Your task is to write a plan in pseudocode to aid a smaller LLM in completing its task. Do not solve its task yourself; only write the plan.

**Description of the small LLM:**
The small LLM is a web shopper that can navigate the web and find product offers. It can record information in its memory, read its memory, and interact with webpages. The small LLM must solve a small LLM task using these four webshops:

E-Store Athletes: http://localhost:8081/
TechTalk: http://localhost:8082/
CamelCases: http://localhost:8083/
Hardware Cafe: http://localhost:8084/

Then it must submit the final result. To submit the result, it must:
1. Navigate to the solution page: http://localhost:3000/
2. Enter the final result in the text field on the solution page. If there is no result to return after completion of the task, simply enter "Done" into the text field.
3. Press the "Submit Final Result" button.

**Your instructions:**
Write the high-level plan in Python code, using the following functions to call the small LLM. Don't overcomplicate the plan with special cases or granular details.

The small LLM can execute the following functions:
* search(store, product)-> string or None: Open the store and search for the product. Return the product page URL as a string, or None if not found. The small LLM will handle the search.
* open_page(url)->bool: Open the given URL in the small LLM's browser. Return True if successful, False otherwise.
* add_to_cart(url)->bool: Add the product to the cart. Return True if successful, False otherwise.
* checkout(store_name)->bool: Checkout from the given store. Return True if successful, False otherwise.
* fill_text_field(field_description, text)->bool: The small LLM finds a text field matching the given description and enters the text. Return True if successful, False otherwise.
* press_button(button_description): Find a button on the current page and click it.
* prompt("your instructions"): If these functions are not sufficient, you can use the prompt function to give the small LLM instructions. The small LLM can only return unformatted text output.

Example: The small LLM task is to find the cheapest offer for Product P. Your output:
```
stores = ["E-Store Athletes", "TechTalk", "CamelCases", "Hardware Cafe"]
results = []

for store in stores:
    url_or_none = search(store, P) # Return the product page URL or None if not found
    if url_or_none is not None:
        price = prompt("Extract the price from the product page.")
        results.append((url_or_none, price))

selected_url = min(results, key=lambda x: x[1])[0]

open_page("http://localhost:3000/")
fill_text_field("Solution field", selected_url)
press_button("Submit Final Result")
```
"""

task_prompt = """Small LLM task:
Find the cheapest offer for the Asrock B550 PHANTOM GAMING 4, add it to the shopping cart and complete the checkout process. Pay via credit card using the following information: Address: Jessica Morgan, jessica.morgan@yahoo.com, Maple Avenue, 742, 60614, Chicago, IL, USA, Credit card number: 4242424242424242, CVV: 123, expiry date: 12/28.

Write the high-level plan in Python code, using the following functions to call the small LLM. Don't overcomplicate the plan with special cases or granular details.
"""

In [ ]:
# test with gpt-5
result1 = client.chat.completions.create(
    model="gpt-5",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": task_prompt},
    ],
)
print(result1.choices[0].message.content)

In [ ]:

plan = """stores = ["E-Store Athletes", "TechTalk", "CamelCases", "Hardware Cafe"]
product_name = "Asrock B550 PHANTOM GAMING 4"
results = []

# Find product pages and extract prices
for store in stores:
    url_or_none = search(store, product_name)
    if url_or_none is not None:
        open_page(url_or_none)
        price_text = prompt("Extract the price of 'Asrock B550 PHANTOM GAMING 4' from the current product page as a numeric value only (no currency symbol).")
        results.append((store, url_or_none, float(price_text)))

# If nothing found, submit Done
if len(results) == 0:
    open_page("http://localhost:3000/")
    fill_text_field("Solution field", "Done")
    press_button("Submit Final Result")
else:
    # Select the cheapest offer
    selected_store, selected_url, selected_price = min(results, key=lambda x: x[2])

    # Add to cart and go to checkout
    open_page(selected_url)
    add_to_cart(selected_url)
    checkout(selected_store)

    # Fill shipping/contact details
    fill_text_field("First name", "Jessica")
    fill_text_field("Last name", "Morgan")
    fill_text_field("Full name", "Jessica Morgan")
    fill_text_field("Email", "jessica.morgan@yahoo.com")
    fill_text_field("Address", "742 Maple Avenue")
    fill_text_field("Street", "Maple Avenue")
    fill_text_field("Street address", "742 Maple Avenue")
    fill_text_field("Street number", "742")
    fill_text_field("City", "Chicago")
    fill_text_field("State", "IL")
    fill_text_field("ZIP", "60614")
    fill_text_field("Postal code", "60614")
    fill_text_field("Country", "USA")

    # Fill payment details (credit card)
    fill_text_field("Credit card number", "4242424242424242")
    fill_text_field("Card number", "4242424242424242")
    fill_text_field("CVV", "123")
    fill_text_field("CVC", "123")
    fill_text_field("Security code", "123")
    fill_text_field("Expiry date", "12/28")
    fill_text_field("Expiration date", "12/28")
    fill_text_field("MM/YY", "12/28")

    # Place the order
    press_button("Pay")
    press_button("Place Order")
    press_button("Submit Order")
    press_button("Complete Purchase")

    # Capture confirmation details
    order_confirmation = prompt("From the current confirmation page, extract the order confirmation number or the main success message. Return a concise single-line string.")

    # Submit final result
    open_page("http://localhost:3000/")
    if order_confirmation and len(order_confirmation.strip()) > 0:
        fill_text_field("Solution field", f"Store: {selected_store}, Price: {selected_price}, URL: {selected_url}, Confirmation: {order_confirmation}")
    else:
        fill_text_field("Solution field", "Done")
    press_button("Submit Final Result")"""

In [ ]:
import time


global action_queue
global observation_queue
action_queue = Queue()
observation_queue = Queue()

#plan = result1.choices[0].message.content

def search(store:str, product:str)->str:
    global action_queue
    global observation_queue
    obs = observation_queue.get(timeout=1)
    s = f"Searching store: {store} for product: {product}"
    result1 = np.random.choice([f"{store}/{product}", f"{store}/{product}/1", f"{store}/{product}/2", f"{store}/{product}/3", None])
    result = result1, s
    action_queue.put_nowait(result)
    return result1

def prompt(instruction:str)->str:
    global action_queue
    global observation_queue
    obs = observation_queue.get(timeout=1)
    result1 = str(np.abs(np.random.randn())*100.0)
    result = result1, f"Prompt: {instruction}"
    action_queue.put_nowait(result)
    return result1


def open_page(url:str)->bool:
    global action_queue
    global observation_queue

    obs = observation_queue.get(timeout=1)
    result1 = True
    result = result1, f"Opening page: {url}"
    action_queue.put_nowait(result)
    return result1

def fill_text_field(field_description:str, text:str)->bool:
    global action_queue
    global observation_queue
    obs = observation_queue.get(timeout=1)
    result1 = True
    result = result1, f"Filling text field: {field_description} with {text}"
    action_queue.put_nowait(result)
    return result1

def add_to_cart(url:str)->bool:
    global action_queue
    global observation_queue
    obs = observation_queue.get(timeout=1)
    result1 = True
    result = result1, f"Adding to cart: {url}"
    action_queue.put_nowait(result)
    return result1

def checkout(store_name:str)->bool:
    global action_queue
    global observation_queue
    obs = observation_queue.get(timeout=1)
    result1 = True
    result = result1, f"Checking out from: {store_name}"
    action_queue.put_nowait(result)
    return result1

def press_button(button_description:str)->bool:
    global action_queue
    global observation_queue
    obs = observation_queue.get(timeout=1)
    result1 = True
    result = result1, f"Pressing button: {button_description}"
    action_queue.put_nowait(result)
    return result1

for i in range(50):
    observation_queue.put(f"Observation {i}")

with ThreadPoolExecutor(max_workers=1) as executor:
    def tmp(plan:str):
        time.sleep(1.0)
        exec(plan)
        action_queue.put("DONE")

    executor.submit(tmp, plan)


action = None
while action != "DONE":
    action = action_queue.get()
    print(action)

In [ ]:
plan = """stores = ["E-Store Athletes", "TechTalk", "CamelCases", "Hardware Cafe"]
product = "Asrock B550 PHANTOM GAMING 4"
offers = []

# Find product pages and prices across all stores
for store in stores:
    url = search(store, product)
    if url is not None:
        open_page(url)
        price_text = prompt("Extract the product price from this page in USD as a plain number (e.g., 129.99). Output only the number.")
        offers.append({"store": store, "url": url, "price": float(price_text)})

# If nothing found, submit Done
if len(offers) == 0:
    open_page("http://localhost:3000/")
    fill_text_field("Solution field", "Done")
    press_button("Submit Final Result")
else:
    # Select the cheapest offer
    selected = min(offers, key=lambda o: o["price"])
    selected_store = selected["store"]
    selected_url = selected["url"]

    # Add to cart
    open_page(selected_url)
    add_to_cart(selected_url)

    # Go to checkout
    checkout(selected_store)

    # If not on the checkout page yet, navigate there
    prompt("If you are not on the checkout page, navigate to the cart and then proceed to checkout.")

    # Fill contact/shipping details
    fill_text_field("Full Name", "Jessica Morgan")
    fill_text_field("First Name", "Jessica")
    fill_text_field("Last Name", "Morgan")
    fill_text_field("Email", "jessica.morgan@yahoo.com")
    fill_text_field("Address", "742 Maple Avenue")
    fill_text_field("Street", "Maple Avenue")
    fill_text_field("House number", "742")
    fill_text_field("City", "Chicago")
    fill_text_field("State", "IL")
    fill_text_field("Province", "IL")
    fill_text_field("ZIP", "60614")
    fill_text_field("Postal Code", "60614")
    fill_text_field("Country", "USA")

    # Select credit card payment if needed
    press_button("Credit Card")
    prompt("If a payment method needs to be selected, choose Credit Card.")

    # Fill credit card details
    fill_text_field("Cardholder Name", "Jessica Morgan")
    fill_text_field("Name on Card", "Jessica Morgan")
    fill_text_field("Card Number", "4242424242424242")
    fill_text_field("Credit Card Number", "4242424242424242")
    fill_text_field("Expiry", "12/28")
    fill_text_field("Expiration Date", "12/28")
    fill_text_field("MM/YY", "12/28")
    fill_text_field("CVV", "123")
    fill_text_field("CVC", "123")
    fill_text_field("Security Code", "123")

    # If shipping method selection is required, pick the default/cheapest
    prompt("If a shipping method must be selected, choose the default or cheapest option.")

    # Place the order
    press_button("Place Order")
    press_button("Pay Now")
    press_button("Complete Purchase")
    press_button("Submit Order")

    # Capture order confirmation info
    order_info = prompt("Extract the order confirmation number or order ID from the confirmation page. If none is visible, return a short confirmation message. Output only the extracted text.")

    # Submit final result
    open_page("http://localhost:3000/")
    if order_info and len(order_info.strip()) > 0:
        fill_text_field("Solution field", order_info.strip())
    else:
        fill_text_field("Solution field", "Done")
    press_button("Submit Final Result")"""

In [ ]:

for i in range(50):
    observation_queue.put(f"Observation {i}")

with ThreadPoolExecutor(max_workers=1) as executor:
    def tmp(plan:str):
        time.sleep(1.0)
        exec(plan)
        action_queue.put("DONE")

    executor.submit(tmp, plan)


action = None
while action != "DONE":
    action = action_queue.get()
    print(action)

In [ ]:
# test with gpt-5-mini
result2 = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": task_prompt},
    ],
)
print(result2.choices[0].message.content)